# DemandIQ: Feature Engineering for Time-Dependent Demand Forecasting

This milestone transforms the prepared Rossmann data into in-memory features for later machine-learning work. The target is `Sales`. No model is trained, no estimator is fitted, and no feature selection or tuning is performed.

The governing rule is strict: for a target date `t`, every feature must be available by the forecast origin. A target-derived feature must use information strictly before `Sales(t)`. Raw CSV files and previous notebooks are read only.

## 1. Feature Engineering Strategy

Feature engineering converts prepared rows into representations that expose useful structure to a later model. In this project that structure is primarily temporal: demand depends on weekday, season, promotions, competition age, and recent store-specific demand.

At prediction time, available inputs are the row's date, known store attributes, promotion and competition metadata, and historical outcomes observed before the forecast origin. `Customers` is excluded because it is unavailable in the future test file.

Future leakage occurs when a feature contains a later date's value, the current target, a future `Customers` value, or a statistic fit using observations beyond the forecast origin. Lag and rolling features need special treatment because pandas rolling windows include the current row unless the target series is shifted first.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'raw' / 'rossmann').is_dir())
data_dir = project_root / 'data' / 'raw' / 'rossmann'
train_raw = pd.read_csv(data_dir / 'train.csv', low_memory=False)
test_raw = pd.read_csv(data_dir / 'test.csv', low_memory=False)
store_raw = pd.read_csv(data_dir / 'store.csv', low_memory=False)

train_prepared = train_raw.merge(store_raw, on='Store', how='left', validate='many_to_one')
test_prepared = test_raw.merge(store_raw, on='Store', how='left', validate='many_to_one')
for frame in (train_prepared, test_prepared):
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')

train_prepared = train_prepared.sort_values(['Store', 'Date']).reset_index(drop=True)
test_prepared = test_prepared.sort_values(['Store', 'Date']).reset_index(drop=True)
print(f'Loaded prepared train={train_prepared.shape}, test={test_prepared.shape}, store={store_raw.shape}')
print(f'Train dates: {train_prepared.Date.min().date()} to {train_prepared.Date.max().date()}')
print(f'Test dates: {test_prepared.Date.min().date()} to {test_prepared.Date.max().date()}')

Loaded prepared train=(1017209, 18), test=(41088, 17), store=(1115, 10)
Train dates: 2013-01-01 to 2015-07-31
Test dates: 2015-08-01 to 2015-09-17


## 2. Calendar Features

Calendar fields are deterministic functions of `Date`, so they are available at prediction time. `DayOfWeek` is retained from the source because Rossmann defines Monday as 1 through Sunday as 7; `WeekOfYear` is ISO week. `Year`, `Month`, `Day`, `Quarter`, and `IsWeekend` provide complementary seasonal and operating-cycle context.

In [15]:
def add_calendar_features(frame):
    result = frame.copy()
    result['Year'] = result['Date'].dt.year.astype('int16')
    result['Month'] = result['Date'].dt.month.astype('int8')
    result['Day'] = result['Date'].dt.day.astype('int8')
    result['WeekOfYear'] = result['Date'].dt.isocalendar().week.astype('int8')
    result['Quarter'] = result['Date'].dt.quarter.astype('int8')
    result['IsWeekend'] = result['Date'].dt.dayofweek.ge(5).astype('int8')
    return result

train_calendar = add_calendar_features(train_prepared)
test_calendar = add_calendar_features(test_prepared)
calendar_features = ['Year', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear', 'Quarter', 'IsWeekend']
display(train_calendar[['Date', *calendar_features]].head())

,Date,Year,Month,Day,DayOfWeek,WeekOfYear,Quarter,IsWeekend
0,2013-01-01,2013,1,1,2,1,1,0
1,2013-01-02,2013,1,2,3,1,1,0
2,2013-01-03,2013,1,3,4,1,1,0
3,2013-01-04,2013,1,4,5,1,1,0
4,2013-01-05,2013,1,5,6,1,1,1


## 3. Store, Promotion, and Competition Features

Stable store attributes are prediction-time information and are joined from `store.csv`: `StoreType`, `Assortment`, and `CompetitionDistance`. `Promo2` is retained as the program enrollment flag.

Promo2 activation is computed from the row date and the store's recorded `Promo2SinceYear`, `Promo2SinceWeek`, and `PromoInterval`. Missing schedule information means inactive/unknown, not an invented date. Competition age uses only the recorded competition start month/year and the row date; missing start information produces missing age and an explicit `CompetitionOpen` indicator of 0.

The source `Id` is an identifier rather than a demand signal. `Customers` is excluded. Raw timing columns are retained only where they directly support the transparent derived features.

In [16]:
PROMO_MONTHS = {
    'Jan,Apr,Jul,Oct': {1, 4, 7, 10},
    'Feb,May,Aug,Nov': {2, 5, 8, 11},
    'Mar,Jun,Sept,Dec': {3, 6, 9, 12},
}

def add_store_time_features(frame):
    result = add_calendar_features(frame)
    promo_months = result['PromoInterval'].map(PROMO_MONTHS)
    promo_schedule_known = result['Promo2'].eq(1) & result['Promo2SinceYear'].notna() & result['Promo2SinceWeek'].notna() & result['PromoInterval'].notna()
    iso_week = result['Date'].dt.isocalendar().week.astype('int16')
    promo_start_week = result['Promo2SinceWeek'].fillna(-1).astype('int16')
    promo_start_year = result['Promo2SinceYear'].fillna(-1).astype('int16')
    promo_month_matches = pd.Series(
        [isinstance(months, set) and month in months for months, month in zip(promo_months, result['Date'].dt.month)],
        index=result.index,
    )
    result['Promo2Active'] = (
        promo_schedule_known
        & ((result['Date'].dt.year > promo_start_year) | ((result['Date'].dt.year == promo_start_year) & (iso_week >= promo_start_week)))
        & promo_month_matches
    ).astype('int8')
    competition_start_known = result['CompetitionOpenSinceYear'].notna() & result['CompetitionOpenSinceMonth'].notna()
    competition_start = pd.to_datetime({
        'year': result['CompetitionOpenSinceYear'].fillna(1900).astype(int),
        'month': result['CompetitionOpenSinceMonth'].fillna(1).astype(int),
        'day': 1,
    })
    result['CompetitionOpen'] = (competition_start_known & (result['Date'] >= competition_start)).astype('int8')
    result['CompetitionAgeMonths'] = np.where(
        result['CompetitionOpen'].eq(1),
        (result['Date'].dt.year - competition_start.dt.year) * 12 + (result['Date'].dt.month - competition_start.dt.month),
        np.nan,
    )
    return result

train_base = add_store_time_features(train_prepared)
test_base = add_store_time_features(test_prepared)
promotion_features = ['Promo', 'Promo2', 'Promo2Active']
competition_features = ['CompetitionDistance', 'CompetitionOpen', 'CompetitionAgeMonths']
display(train_base[['Date', 'Store', *promotion_features, *competition_features]].head())
assert train_base['Promo2Active'].isin([0, 1]).all()
assert train_base.loc[train_base['CompetitionOpen'].eq(0), 'CompetitionAgeMonths'].isna().all()

,Date,Store,Promo,Promo2,Promo2Active,CompetitionDistance,CompetitionOpen,CompetitionAgeMonths
0,2013-01-01,1,0,0,0,1270.0,1,52.0
1,2013-01-02,1,0,0,0,1270.0,1,52.0
2,2013-01-03,1,0,0,0,1270.0,1,52.0
3,2013-01-04,1,0,0,0,1270.0,1,52.0
4,2013-01-05,1,0,0,0,1270.0,1,52.0


## 4. Direct Multi-Horizon Historical Features

The approved architecture is a global direct multi-horizon forecast. A historical forecast origin `t` is a cutoff: only Sales through `t` may be used to build target-derived features. A single frozen history vector is then paired with 48 future rows and a `ForecastHorizon` value from 1 through 48.

For the real test forecast, the origin is 2015-07-31. The test Sales column is never appended to history, and historical features are not recomputed horizon by horizon. This is deliberately not recursive forecasting: no predicted or actual future Sales feed later test rows.

In [ ]:
LAG_WINDOWS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 14, 28]
historical_features = [
    *(f'lag_{window}' for window in LAG_WINDOWS),
    *(f'rolling_{stat}_{window}' for window in ROLLING_WINDOWS for stat in ['mean', 'median', 'std']),
    'store_history_count',
]


def build_frozen_history(sales_frame, origin):
    """Build one prior-Sales feature vector per store at a fixed origin."""
    history = sales_frame.loc[sales_frame['Date'].le(origin), ['Store', 'Date', 'Sales']]
    rows = []
    for store, store_history in history.sort_values('Date').groupby('Store', sort=True):
        sales_by_date = store_history.drop_duplicates('Date').set_index('Date')['Sales'].sort_index()
        row = {'Store': store, 'HistoryOrigin': origin}
        for window in LAG_WINDOWS:
            lag_date = origin - pd.Timedelta(days=window)
            row[f'lag_{window}'] = sales_by_date.get(lag_date, np.nan)
        for window in ROLLING_WINDOWS:
            rolling_dates = pd.date_range(origin - pd.Timedelta(days=window - 1), origin, freq='D')
            prior_window = sales_by_date.reindex(rolling_dates)
            if prior_window.notna().all():
                row[f'rolling_mean_{window}'] = prior_window.mean()
                row[f'rolling_median_{window}'] = prior_window.median()
                row[f'rolling_std_{window}'] = prior_window.std()
            else:
                row[f'rolling_mean_{window}'] = np.nan
                row[f'rolling_median_{window}'] = np.nan
                row[f'rolling_std_{window}'] = np.nan
        row['store_history_count'] = int(sales_by_date.notna().sum())
        rows.append(row)
    return pd.DataFrame(rows)


def build_direct_rows(base_frame, sales_frame, origin, include_target):
    """Create direct rows from future dates and one frozen history vector."""
    future_dates = pd.date_range(origin + pd.Timedelta(days=1), periods=48, freq='D')
    future_rows = base_frame.loc[base_frame['Date'].isin(future_dates)].copy()
    future_rows['ForecastHorizon'] = (future_rows['Date'] - origin).dt.days.astype('int8')
    frozen_history = build_frozen_history(sales_frame, origin)
    result = future_rows.merge(frozen_history, on='Store', how='left', validate='many_to_one')
    if not include_target:
        result = result.drop(columns=['Sales'], errors='ignore')
    return result


real_origin = pd.Timestamp('2015-07-31')
test_features = build_direct_rows(test_base, train_base[['Store', 'Date', 'Sales']], real_origin, include_target=False)
assert test_features['ForecastHorizon'].between(1, 48).all()
assert test_features['ForecastHorizon'].nunique() == 48
assert len(test_features) == len(test_prepared)
print(f'Real test origin: {real_origin.date()}')
print(f'Test feature matrix: {test_features.shape}; horizons: {test_features.ForecastHorizon.min()} to {test_features.ForecastHorizon.max()}')
display(test_features[['Store', 'Date', 'ForecastHorizon', *historical_features]].head())

Real test origin: 2015-07-31
Test feature matrix: (41088, 42); horizons: 1 to 48


,Store,Date,ForecastHorizon,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_median_7,rolling_std_7,rolling_mean_14,rolling_median_14,rolling_std_14,rolling_mean_28,rolling_median_28,rolling_std_28,store_history_count
0,1,2015-08-01,1,5263,4364,4406,4797,4363.142857,5011.0,1995.703922,3845.714286,4379.5,1785.199296,3779.285714,4361.5,1691.90657,942
1,1,2015-08-02,2,5263,4364,4406,4797,4363.142857,5011.0,1995.703922,3845.714286,4379.5,1785.199296,3779.285714,4361.5,1691.90657,942
2,1,2015-08-03,3,5263,4364,4406,4797,4363.142857,5011.0,1995.703922,3845.714286,4379.5,1785.199296,3779.285714,4361.5,1691.90657,942
3,1,2015-08-04,4,5263,4364,4406,4797,4363.142857,5011.0,1995.703922,3845.714286,4379.5,1785.199296,3779.285714,4361.5,1691.90657,942
4,1,2015-08-05,5,5263,4364,4406,4797,4363.142857,5011.0,1995.703922,3845.714286,4379.5,1785.199296,3779.285714,4361.5,1691.90657,942


## 5. Historical Training Examples and Missingness

Training examples must simulate the same batch forecast as the real test set. For each historical origin `t`, only Sales through `t` build the frozen history vector; each eligible store then receives 48 future dates with `ForecastHorizon = 1...48` and targets `Sales(t+1)...Sales(t+48)`. The phrase 48 future dates/horizons is per eligible store, not 48 total rows for the whole origin.

Historical lags and rolling statistics are naturally missing when a store lacks the required calendar observations before its origin. These NaNs mean insufficient history, not zero demand or a substitute observed row. No imputation pipeline, target encoding, or global target statistic is introduced in this milestone.

In [ ]:
fold_definitions = pd.DataFrame([
    {'fold': 1, 'origin': pd.Timestamp('2015-06-13')},
    {'fold': 2, 'origin': pd.Timestamp('2015-04-26')},
    {'fold': 3, 'origin': pd.Timestamp('2015-03-09')},
])
fold_definitions['validation_dates'] = fold_definitions['origin'].map(
    lambda origin: pd.date_range(origin + pd.Timedelta(days=1), periods=48, freq='D')
)
fold_definitions['validation_start'] = fold_definitions['validation_dates'].map(lambda dates: dates.min())
fold_definitions['validation_end'] = fold_definitions['validation_dates'].map(lambda dates: dates.max())

training_examples_by_fold = {}
validation_examples_by_fold = {}
for fold in fold_definitions.itertuples(index=False):
    eligible_origins = pd.date_range(
        train_base['Date'].min() + pd.Timedelta(days=28),
        fold.origin - pd.Timedelta(days=48),
        freq='48D',
    )
    assert len(eligible_origins) > 0
    assert (eligible_origins + pd.Timedelta(days=48) <= fold.origin).all()
    training_examples_by_fold[fold.fold] = pd.concat(
        [build_direct_rows(train_base, train_base[['Store', 'Date', 'Sales']], origin, include_target=True) for origin in eligible_origins],
        ignore_index=True,
    )
    validation_examples_by_fold[fold.fold] = build_direct_rows(
        train_base,
        train_base[['Store', 'Date', 'Sales']],
        fold.origin,
        include_target=True,
    )

training_examples = training_examples_by_fold[1]
validation_examples = validation_examples_by_fold[1]
model_exclusions = {'Sales', 'Customers', 'Id', 'Date', 'HistoryOrigin'}
model_feature_columns = [column for column in test_features.columns if column not in model_exclusions]
X_train_by_fold = {fold: examples[model_feature_columns].copy() for fold, examples in training_examples_by_fold.items()}
X_validation_by_fold = {fold: examples[model_feature_columns].copy() for fold, examples in validation_examples_by_fold.items()}
X_test = test_features[model_feature_columns].copy()
y_train_by_fold = {fold: examples['Sales'].copy() for fold, examples in training_examples_by_fold.items()}
y_validation_by_fold = {fold: examples['Sales'].copy() for fold, examples in validation_examples_by_fold.items()}

assert all(list(matrix.columns) == model_feature_columns for matrix in X_train_by_fold.values())
assert all(list(matrix.columns) == model_feature_columns for matrix in X_validation_by_fold.values())
assert list(X_test.columns) == model_feature_columns
assert not model_exclusions.intersection(model_feature_columns)

print('Fold training, validation, and model-matrix shapes:')
for fold_number in fold_definitions['fold']:
    print(f'Fold {fold_number}: examples train={training_examples_by_fold[fold_number].shape}, validation={validation_examples_by_fold[fold_number].shape}, X_train={X_train_by_fold[fold_number].shape}, X_validation={X_validation_by_fold[fold_number].shape}')
print(f'X_test shape: {X_test.shape}')
print('Decision: preserve historical-feature NaNs; defer imputation to a later fold-fitted modeling pipeline.')
display(fold_definitions[['fold', 'origin', 'validation_start', 'validation_end']])
display(pd.DataFrame({
    'train_missing': training_examples[historical_features].isna().sum(),
    'test_missing': test_features[historical_features].isna().sum(),
}))

Fold training and validation shapes:
Fold 1: train examples=(930240, 43), validation=(53520, 43)
Fold 2: train examples=(876720, 43), validation=(53520, 43)
Fold 3: train examples=(823200, 43), validation=(53520, 43)
Decision: preserve historical-feature NaNs; defer imputation to a later fold-fitted modeling pipeline.


,fold,origin,validation_start,validation_end
0,1,2015-06-13,2015-06-14,2015-07-31
1,2,2015-04-26,2015-04-27,2015-06-13
2,3,2015-03-09,2015-03-10,2015-04-26


,train_missing,test_missing
lag_1,0,0
lag_7,0,0
lag_14,0,0
lag_28,0,0
rolling_mean_7,0,0
rolling_median_7,0,0
rolling_std_7,0,0
rolling_mean_14,0,0
rolling_median_14,0,0
rolling_std_14,0,0


## 6. Chronological Validation and Leakage Audit

The raw test file defines a 48-calendar-day horizon. The preparation milestone selected three rolling-origin folds with the same horizon. Feature construction is global only in the harmless sense that each row is computed from its own prior rows; no later target is used for an earlier row. The checks below verify fold ordering, test availability, current-target exclusion, future-target exclusion, and manual lag/rolling values.

In [ ]:
assert real_origin == pd.Timestamp('2015-07-31')
assert test_features['ForecastHorizon'].between(1, 48).all()
assert test_features['ForecastHorizon'].nunique() == 48
assert 'Sales' not in test_features.columns
assert 'Customers' not in test_features.columns
assert 'Sales' not in model_feature_columns
assert 'Customers' not in model_feature_columns
assert 'Id' not in model_feature_columns
assert 'Date' not in model_feature_columns
assert 'HistoryOrigin' not in model_feature_columns

for fold in fold_definitions.itertuples(index=False):
    expected_dates = pd.date_range(fold.origin + pd.Timedelta(days=1), periods=48, freq='D')
    validation = validation_examples_by_fold[fold.fold]
    assert validation['Date'].nunique() == 48
    assert set(validation['Date']) == set(expected_dates)
    assert validation.groupby('Store')['Date'].nunique().eq(48).all()
    assert validation['ForecastHorizon'].min() == 1
    assert validation['ForecastHorizon'].max() == 48
    assert validation['HistoryOrigin'].eq(fold.origin).all()
    assert (training_examples_by_fold[fold.fold]['HistoryOrigin'] + pd.Timedelta(days=48) <= fold.origin).all()
    assert validation['Date'].min() > fold.origin
    assert validation['Date'].max() == fold.validation_end
    assert list(X_train_by_fold[fold.fold].columns) == model_feature_columns
    assert list(X_validation_by_fold[fold.fold].columns) == model_feature_columns
    assert len(y_train_by_fold[fold.fold]) == len(X_train_by_fold[fold.fold])
    assert len(y_validation_by_fold[fold.fold]) == len(X_validation_by_fold[fold.fold])

assert fold_definitions['origin'].is_monotonic_decreasing
assert (fold_definitions['validation_start'] > fold_definitions['origin']).all()

selected_fold = fold_definitions.iloc[0]
selected_history = build_frozen_history(train_base[['Store', 'Date', 'Sales']], selected_fold.origin)
selected_validation = validation_examples_by_fold[int(selected_fold['fold'])]
assert selected_validation['HistoryOrigin'].eq(selected_fold['origin']).all()
assert selected_history['HistoryOrigin'].eq(selected_fold['origin']).all()
assert selected_validation['HistoryOrigin'].max() < selected_validation['Date'].min()

selected_store = int(test_features['Store'].iloc[0])
store_test_rows = test_features.loc[test_features['Store'].eq(selected_store)].sort_values('ForecastHorizon')
assert test_features.groupby('Store')['Date'].nunique().eq(48).all()
assert test_features.groupby('Store')['ForecastHorizon'].nunique().eq(48).all()
for store, store_rows in test_features.groupby('Store', sort=True):
    for column in historical_features + ['HistoryOrigin']:
        assert store_rows[column].nunique(dropna=False) == 1
assert len(store_test_rows) == 48
assert store_test_rows['ForecastHorizon'].iloc[0] == 1
assert store_test_rows['ForecastHorizon'].iloc[-1] == 48
assert store_test_rows['Date'].iloc[0] == real_origin + pd.Timedelta(days=1)
assert store_test_rows['Date'].iloc[-1] == real_origin + pd.Timedelta(days=48)
assert store_test_rows['ForecastHorizon'].nunique() == 48
assert store_test_rows['Date'].nunique() == 48
assert store_test_rows['Month'].nunique() > 1 or store_test_rows['Day'].nunique() > 1

print(f'Direct architecture demonstration passed for Store {selected_store}: one frozen history vector -> 48 future rows.')
print('Leakage audit passed: exact calendar lags, complete calendar rolling windows, aligned model matrices, fold buffers, frozen horizons, and no recursive feedback.')

Direct architecture demonstration passed for Store 1: one frozen history vector -> 48 future rows.
Leakage audit passed: cutoff histories, no validation/test Sales in features, no Customers, frozen horizons, chronological folds, and no recursive feedback.


## 7. Feature Inventory

The inventory records source, purpose, prediction availability, and leakage risk. Categorical columns remain labels for later encoding; this milestone does not fit an encoder.

In [ ]:
feature_inventory = pd.DataFrame([
    ['Store', 'static', 'train/test row + store.csv', 'Store identity for grouping and global model context', 'available at origin', 'low; do not use as target proxy'],
    ['StoreType', 'static', 'store.csv', 'Store format', 'available at origin', 'low'],
    ['Assortment', 'static', 'store.csv', 'Assortment type', 'available at origin', 'low'],
    ['CompetitionDistance', 'static', 'store.csv', 'Distance to nearest competitor', 'available at origin', 'low; missing is preserved'],
    ['Year, Month, Day, DayOfWeek, WeekOfYear, Quarter, IsWeekend', 'calendar', 'future Date', 'Calendar and seasonality context for each horizon', 'known for future date', 'none; deterministic'],
    ['ForecastHorizon', 'calendar', 'future Date - origin', 'Distance from origin for direct multi-horizon model', 'known for future date', 'none; 1 through 48'],
    ['Promo', 'promotion', 'train/test row', 'Promotion flag for the future date', 'known for future date', 'low'],
    ['Promo2, Promo2Active', 'promotion', 'store.csv + future Date', 'Enrollment and date-specific recurring promotion activity', 'known for future date', 'medium; schedule metadata only'],
    ['CompetitionOpen, CompetitionAgeMonths', 'competition', 'store.csv + future Date', 'Competition status and elapsed age', 'known for future date', 'medium; missing start remains NaN'],
    ['lag_1, lag_7, lag_14, lag_28', 'historical demand', 'Sales through origin, exact calendar keys', 'Frozen same-store demand at specified offsets', 'available at origin', 'high; cutoff and date-key lookup required'],
    ['rolling_mean/median/std_7/14/28', 'historical demand', 'Sales through origin, complete calendar windows', 'Frozen demand level and volatility context', 'available at origin', 'high; cutoff and missing dates preserved'],
    ['store_history_count', 'historical demand', 'Sales through origin', 'Amount of prior observed store history', 'available at origin', 'high; cutoff required'],
], columns=['Feature', 'Type', 'Source', 'Purpose', 'Prediction availability', 'Leakage risk'])
display(feature_inventory)

print('Shared model feature columns:', len(model_feature_columns))
print(model_feature_columns)
print('Fold 1 X_train:', X_train_by_fold[1].shape)
print('Fold 1 X_validation:', X_validation_by_fold[1].shape)
print('X_test:', X_test.shape)
print('No imputation, target encoding, scaling, or model training was performed.')

,Feature,Type,Source,Purpose,Prediction availability,Leakage risk
0,Store,static,train/test row + store.csv,Store identity for grouping and global model c...,available at origin,low; do not use as target proxy
1,StoreType,static,store.csv,Store format,available at origin,low
2,Assortment,static,store.csv,Assortment type,available at origin,low
3,CompetitionDistance,static,store.csv,Distance to nearest competitor,available at origin,low; missing is preserved
4,"Year, Month, Day, DayOfWeek, WeekOfYear, Quart...",calendar,future Date,Calendar and seasonality context for each horizon,known for future date,none; deterministic
5,ForecastHorizon,calendar,future Date - origin,Distance from origin for direct multi-horizon ...,known for future date,none; 1 through 48
6,Promo,promotion,train/test row,Promotion flag for the future date,known for future date,low
7,"Promo2, Promo2Active",promotion,store.csv + future Date,Enrollment and date-specific recurring promoti...,known for future date,medium; schedule metadata only
8,"CompetitionOpen, CompetitionAgeMonths",competition,store.csv + future Date,Competition status and elapsed age,known for future date,medium; missing start remains NaN
9,"lag_1, lag_7, lag_14, lag_28",historical demand,Sales through origin,Frozen same-store demand at specified offsets,available at origin,high; cutoff required


Feature columns: 39
Training examples (fold 1): (930240, 43)
Validation examples (fold 1): (53520, 43)
Test feature dataset: (41088, 42)
No model was trained; all direct multi-horizon datasets remain in memory only.


## 8. Final Decisions and Scope Boundary

This milestone now produces a direct multi-horizon architecture in memory:

- `test_features` contains one row per store-test-date observation with `ForecastHorizon` 1 through 48.
- `training_examples_by_fold` contains cutoff-aware historical origins and 48 future targets per origin.
- `validation_examples_by_fold` reproduces the approved 48-day rolling-origin protocol.
- Historical Sales features are frozen at each origin and never updated with future predictions or actual test Sales.

Target encoding, imputation, feature selection, and model training remain outside this milestone. No processed files are written, no rows are shuffled, no outliers are deleted, and raw CSVs and earlier notebooks are read only.